# Multi-step event generation in EasyTPP: the padding fix and intensity-free thinning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ant-research/EasyTemporalPointProcess/blob/main/notebooks/easytpp_multistep_generation.ipynb)

This notebook explains and verifies the fixes for
[issue #13](https://github.com/ant-research/EasyTemporalPointProcess/issues/13),
which raised three questions about event generation:

1. **Why did multi-step generation produce events clustered near the start?**
   Two independent causes: a missing `cumsum` over exponential proposals in the
   thinning sampler (found by the issue reporter, fixed earlier), and a
   **padding bug** in `predict_multi_step_since_last_event` — for any sequence
   shorter than the batch max, generation conditioned on pad events, added
   predicted times to pad values, and even compared against pad labels.
2. **How can `IntensityFree` (IFTPP) generate events when it has no intensity?**
   The log-normal mixture admits a closed-form intensity
   $\lambda(t) = f(t)/S(t)$, now implemented as
   `IntensityFree.compute_intensities_at_sample_times`, making IFTPP fully
   compatible with the thinning sampler and the `gen` pipeline.
3. **What about models without `eval`/`gen` configs?** The `thinning` section
   of the model config is what enables generation; with the fix above, every
   model can now carry one.

We verify each fix numerically below. Requires `easy-tpp` **newer than 0.2.3**
(install from git until the next release).

In [ ]:
import importlib.util
if importlib.util.find_spec('easy_tpp') is None:
    %pip install -q git+https://github.com/ant-research/EasyTemporalPointProcess.git

import torch
import matplotlib.pyplot as plt

from easy_tpp.config_factory import ModelConfig
from easy_tpp.model import TorchIntensityFree, TorchRMTPP
from easy_tpp.model.intensity_free import (
    LogNormalMixtureDistribution, clamp_preserve_gradients)

PAD = 3  # pad token id used throughout

## Setup: a tiny model and a right-padded batch

We build small untrained models (the bugs are structural, so no training is
needed to demonstrate them) and a batch of two sequences with **different true
lengths**: 8 events and 5 events, right-padded to 8. Pads mimic the real
tokenizer: pad type = `pad_token_id`, pad times = pad value.

In [ ]:
def make_model_config(model_id, model_specs=None, num_step_gen=2, **thinning_overrides):
    thinning = {'num_sample': 5, 'num_exp': 100, 'over_sample_rate': 1.5,
                'num_samples_boundary': 5, 'dtime_max': 5, 'patience_counter': 5,
                'num_step_gen': num_step_gen}
    thinning.update(thinning_overrides)
    config = ModelConfig.parse_from_yaml_config({
        'model_id': model_id, 'hidden_size': 8,
        'num_event_types': 3, 'num_event_types_pad': 4, 'event_pad_index': PAD,
        'gpu': -1, 'model_specs': model_specs or {}, 'thinning': thinning,
    })
    config.set('mean_log_inter_time', 0.0)
    config.set('std_log_inter_time', 1.0)
    return config


def make_padded_batch():
    time_delta_seqs = torch.tensor([
        [0.0, 0.4, 0.6, 0.5, 0.7, 0.8, 0.9, 1.0],
        [0.0, 0.3, 0.2, 0.6, 0.4, float(PAD), float(PAD), float(PAD)],
    ])
    time_seqs = torch.tensor([
        [0.0, 0.4, 1.0, 1.5, 2.2, 3.0, 3.9, 4.9],
        [0.0, 0.3, 0.5, 1.1, 1.5, float(PAD), float(PAD), float(PAD)],
    ])
    type_seqs = torch.tensor([
        [0, 1, 2, 0, 1, 2, 0, 1],
        [2, 1, 0, 2, 1, PAD, PAD, PAD],
    ])
    batch_non_pad_mask = torch.tensor([[True] * 8, [True] * 5 + [False] * 3])
    attention_mask = torch.zeros(2, 8, 8, dtype=torch.bool)
    return time_seqs, time_delta_seqs, type_seqs, batch_non_pad_mask, attention_mask

## 1. The padding bug

The pre-fix `predict_multi_step_since_last_event` (copied verbatim below)
discarded `batch_non_pad_mask` and sliced columns from the **tensor end**:
`[:, :-num_step]` for the prefix, `[:, -1:]` for "the last event",
`[:, -num_step-1:]` for the labels. That is only correct for rows whose true
length equals the batch max. For every shorter row it

- conditioned the generation on **pad events** (pad types embedded as history),
- read a **pad value as the last timestamp** and added predicted dtimes to it,
- returned **pad values as the ground-truth labels** to compare against.

This corrupted generated sequences for all but the longest row of each batch —
a second, independent cause of the degenerate generations reported in the
issue (besides the `cumsum` already fixed in the sampler).

In [ ]:
def predict_multi_step_old(model, batch, forward=False):
    """Verbatim copy of the PRE-FIX predict_multi_step_since_last_event."""
    time_seq_label, time_delta_seq_label, event_seq_label, _, _ = batch
    num_step = model.gen_config.num_step_gen
    if not forward:
        time_seq = time_seq_label[:, :-num_step]
        time_delta_seq = time_delta_seq_label[:, :-num_step]
        event_seq = event_seq_label[:, :-num_step]
    else:
        time_seq, time_delta_seq, event_seq = time_seq_label, time_delta_seq_label, event_seq_label
    for _ in range(num_step):
        dtime_boundary = time_delta_seq + model.event_sampler.dtime_max
        accepted_dtimes, weights = model.event_sampler.draw_next_time_one_step(
            time_seq, time_delta_seq, event_seq, dtime_boundary,
            model.compute_intensities_at_sample_times, compute_last_step_only=True)
        dtimes_pred = torch.sum(accepted_dtimes * weights, dim=-1)
        intensities_at_times = model.compute_intensities_at_sample_times(
            time_seq, time_delta_seq, event_seq, dtimes_pred[:, :, None],
            max_steps=event_seq.size()[1])
        intensities_at_times = intensities_at_times.squeeze(dim=-2)
        types_pred = torch.argmax(intensities_at_times, dim=-1)
        types_pred_, dtimes_pred_ = types_pred[:, -1:], dtimes_pred[:, -1:]
        time_pred_ = time_seq[:, -1:] + dtimes_pred_
        time_seq = torch.cat([time_seq, time_pred_], dim=-1)
        time_delta_seq = torch.cat([time_delta_seq, dtimes_pred_], dim=-1)
        event_seq = torch.cat([event_seq, types_pred_], dim=-1)
    return (time_delta_seq[:, -num_step - 1:], event_seq[:, -num_step - 1:],
            time_delta_seq_label[:, -num_step - 1:], event_seq_label[:, -num_step - 1:])

In [ ]:
torch.manual_seed(0)
model = TorchRMTPP(make_model_config('RMTPP'))
batch = make_padded_batch()

with torch.no_grad():
    old = predict_multi_step_old(model, batch)
    new = model.predict_multi_step_since_last_event(batch)

print('short row (true length 5), num_step_gen = 2:')
print(f'  true last 3 real deltas : {batch[1][1, 2:5].tolist()}')
print(f'  true last 3 real types  : {batch[2][1, 2:5].tolist()}')
print(f'  OLD label deltas        : {old[2][1].tolist()}   <- pad values!')
print(f'  OLD label types         : {old[3][1].tolist()}   <- pad values!')
print(f'  NEW label deltas        : {new[2][1].tolist()}')
print(f'  NEW label types         : {new[3][1].tolist()}')

assert torch.equal(new[2][1], batch[1][1, 2:5])
assert torch.equal(new[3][1], batch[2][1, 2:5])
assert torch.all(old[3][1] == PAD)
print('\nPASS: the fix returns real events; the old code returned pads.')

The fix groups rows by their true length (from `batch_non_pad_mask`), runs the
autoregressive generation on **pad-free** slices per group, and scatters the
results back into full-batch tensors — same signature, same return contract,
plus a clear error when a sequence is too short to hold out `num_step_gen`
label events. A regression test
(`tests/test_multi_step_generation.py`) locks this in.

## 2. IntensityFree has a closed-form intensity after all

IFTPP (Shchur et al., ICLR 2020) models the next inter-event time with a
log-normal mixture density $f(t)$ — no intensity appears in training, which is
the point of the paper. For **one-step** prediction EasyTPP always sampled
directly from that density (that is where the paper's RMSE/ACC come from).
But thinning-based **multi-step** generation needs an intensity, and the
mixture admits one in closed form:

$$\lambda(t) = \frac{f(t)}{S(t)}, \qquad \lambda_k(t) = \lambda(t)\,p(k \mid \text{history}),$$

where $S(t)$ is the survival function and marks are conditionally independent
of time. `IntensityFree.compute_intensities_at_sample_times` now implements
exactly this using the distribution's existing `log_prob` and
`log_survival_function`.

**Verification**: with a single mixture component the hazard must equal the
textbook log-normal hazard, which we can compute independently with
`torch.distributions.LogNormal`.

In [ ]:
torch.manual_seed(0)
if_model = TorchIntensityFree(make_model_config(
    'IntensityFree', model_specs={'num_mix_components': 1}))
time_seqs, dts, types = batch[0], batch[1], batch[2]
sample_dtimes = torch.linspace(0.1, 3.0, 20)[None, None, :].expand(2, 8, 20)

with torch.no_grad():
    lambdas = if_model.compute_intensities_at_sample_times(
        time_seqs, dts, types, sample_dtimes)

    # independent closed form: tau ~ LogNormal(loc, scale)
    context = if_model.forward(dts, types)
    raw = if_model.linear(context)
    loc, log_scale = raw[..., 0], raw[..., 1].clamp(-5.0, 3.0)
    ln = torch.distributions.LogNormal(loc[..., None], log_scale.exp()[..., None])
    hazard = ln.log_prob(sample_dtimes).exp() / (1.0 - ln.cdf(sample_dtimes))
    mark_probs = torch.softmax(if_model.mark_linear(context), dim=-1)
    expected = hazard[..., None] * mark_probs[:, :, None, :]

err = (lambdas - expected).abs().max().item()
print(f'shape {tuple(lambdas.shape)} = (batch, seq, samples, marks)')
print(f'all finite & positive: {bool(torch.isfinite(lambdas).all() and (lambdas > 0).all())}')
print(f'max |implementation - closed form| = {err:.2e}')
assert err < 1e-4
print('PASS: implemented intensity matches the closed-form log-normal hazard.')

## 3. Thinning with the new intensity reproduces the model's own density

The strongest end-to-end check: draw next-event times two ways from the *same*
IntensityFree model —

1. **directly** from the log-normal mixture (exact, the model's ground truth), and
2. via the **thinning sampler** driven by the new closed-form intensity.

If the intensity implementation and the sampler (including the `cumsum` fix
from this issue) are correct, the two distributions must agree.

In [ ]:
torch.manual_seed(7)
gen_model = TorchIntensityFree(make_model_config(
    'IntensityFree', model_specs={'num_mix_components': 3},
    num_step_gen=1, num_sample=1, num_exp=500, dtime_max=20))
t1, d1, y1 = batch[0][:1], batch[1][:1], batch[2][:1]

with torch.no_grad():
    # 1. exact samples from the model's own mixture at the last position
    context = gen_model.forward(d1, y1)
    raw = gen_model.linear(context[:, -1:, :])
    k = gen_model.num_mix_components
    dist = LogNormalMixtureDistribution(
        raw[..., :k], clamp_preserve_gradients(raw[..., k:2 * k], -5.0, 3.0),
        torch.log_softmax(raw[..., 2 * k:], dim=-1),
        gen_model.mean_log_inter_time, gen_model.std_log_inter_time)
    direct = dist.sample((4000,)).flatten()

    # 2. thinning samples using compute_intensities_at_sample_times
    thin = []
    for _ in range(400):
        boundary = d1 + gen_model.event_sampler.dtime_max
        accepted, _ = gen_model.event_sampler.draw_next_time_one_step(
            t1, d1, y1, boundary,
            gen_model.compute_intensities_at_sample_times, compute_last_step_only=True)
        thin.append(accepted.flatten())
    thin = torch.cat(thin)

qs = torch.tensor([0.25, 0.5, 0.75, 0.9])
for name, s in [('direct (exact)', direct), ('thinning', thin)]:
    q = torch.quantile(s, qs)
    print(f'{name:>15}: median={q[1]:.3f}  q25={q[0]:.3f}  q75={q[2]:.3f}  q90={q[3]:.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
bins = torch.linspace(0, 8, 60)
ax.hist(direct.clamp(max=8).numpy(), bins=bins.numpy(), density=True, alpha=0.55, label='direct from density (exact)')
ax.hist(thin.clamp(max=8).numpy(), bins=bins.numpy(), density=True, alpha=0.55, label='thinning via closed-form intensity')
ax.set_xlabel('sampled next inter-event time')
ax.set_ylabel('density')
ax.set_title('IntensityFree: thinning matches direct sampling')
ax.legend()
plt.tight_layout()
plt.show()

The distributions coincide (small deficits in the far tail come from the finite
sampling horizon — proposals past `num_exp` draws fall back to `dtime_max`,
which is expected thinning behavior).

## 4. End-to-end multi-step generation

Finally, the full `gen` path on the padded batch — including `IntensityFree`,
which could not run this at all before the fix.

(Note: these models are untrained, so the generated *values* are arbitrary —
random weights can even emit the pad type, whose probability training drives
to zero. What matters here is that the machinery runs, respects true sequence
lengths, and produces finite outputs of the right shape.)

In [ ]:
for name, m in [('RMTPP', TorchRMTPP(make_model_config('RMTPP'))),
                ('IntensityFree', TorchIntensityFree(make_model_config(
                    'IntensityFree', model_specs={'num_mix_components': 3})))]:
    torch.manual_seed(0)
    with torch.no_grad():
        pred_dt, pred_ty, lab_dt, lab_ty = m.predict_multi_step_since_last_event(batch)
    assert pred_dt.shape == (2, 3) and torch.isfinite(pred_dt).all()
    print(f'{name:>14}: generated dtimes (short row) = {[round(x, 3) for x in pred_dt[1].tolist()]}, '
          f'types = {pred_ty[1].tolist()}')
print('\nPASS: both models generate finite multi-step predictions on a padded batch.')

## Summary

| Question from issue #13 | Resolution |
|---|---|
| Events cluster near the start in multi-step generation | Two causes fixed: missing `cumsum` over exponential proposals (credit to the reporter), and the padding bug demonstrated in section 1 |
| How can IFTPP sample without an intensity? | One-step: direct sampling from the density (always the case). Multi-step: closed-form hazard $\lambda = f/S$, verified in sections 2–3 |
| Configs without `eval`/`gen` sections | The `thinning` config enables generation; all models, now including IFTPP, can carry one |

Related: for datasets with large raw inter-event times, also see the opt-in
`data_specs.rescale_time` option (issue #55) — the models' temporal encodings
are not scale-invariant.

### References
- Mei & Eisner. *The Neural Hawkes Process.* NeurIPS 2017 (thinning, Algorithm 2).
- Shchur et al. *Intensity-Free Learning of Temporal Point Processes.* ICLR 2020.
- [Issue #13](https://github.com/ant-research/EasyTemporalPointProcess/issues/13)